# LSTM（長短期記憶，1997）

- **核心原理**：輸入、遺忘、輸出閘門解決梯度消失
- **變體**：Bi-LSTM（雙向）、Stacked LSTM（多層）；peephole LSTM；下一課的 GRU
- **推薦資料集**：IMDB 影評情感分類（50,000 篇英文影評，正面／負面各半；train 25,000、test 25,000）
- **論文**：Hochreiter & Schmidhuber (1997), *Long Short-Term Memory*；Gers, Schmidhuber & Cummins (2000), *Learning to Forget*（遺忘閘門）

本章依照 README 的「每章標準流程」：EDA → Baseline → 手寫模型 → 套件實作 → 調參 → 測試集評估 → 診斷 → 本章小結。

和第 2 課（RNN）的差別：

| | 第 2 課 RNN | 本章 LSTM |
| --- | --- | --- |
| 資料 | 名字：2～20 個字母、18 類 | 影評：中位數約 190 個字、最長上千個字、2 類 |
| 單位 | 字母（字母表 59） | **字**（word；字表取最常見的 20,000 個） |
| 隱藏狀態 | 一個 $h_t$ | 兩個：**cell state** $c_t$（長期記憶）+ $h_t$（輸出） |
| 遞迴 | $h_t = \tanh(x_t W_x + h_{t-1}W_h + b)$ | 三個閘門決定 $c_t$「留多少舊的、加多少新的」、$h_t$「輸出多少」 |
| 梯度往回傳 | 每一步乘 $(1 - h_t^2)\,W_h^\top$（連乘 → 消失／爆炸） | 沿著 $c$ 每一步只乘**遺忘閘門** $f_t$（逐元素、接近 1 時幾乎不衰減） |
| 手寫的層 | `Embedding`、`RNN` | 沿用第 2 課的 `Embedding`、`RNN`（當對照），**新增** `LSTM` |

一句話：**LSTM 把 RNN 的「每一步都整個重寫隱藏狀態」，改成「在一條傳送帶（cell state）上，用閘門決定擦掉多少、寫入多少」—— 傳送帶上的資訊和梯度都可以幾乎不變地走很遠。**

這一章的重點：**四個閘門的公式與直覺、LSTM 的反向傳播（為什麼 $c$ 那條路梯度不會消失）、遺忘閘門偏差初始化成 1 的理由、截斷長文（留開頭還是結尾）、RNN vs LSTM 在長序列上的差距，以及又一個誠實的問題：LSTM 比得過「TF-IDF + 邏輯回歸」嗎？**

### 閱讀指南：本章的符號與跨章引用

下一格是「前置複習」；第 2、3A–3D、4、5 節開頭另有各自的符號表或名詞表。

**資料**：每筆是一篇影評（英文，含 HTML 換行 `<br />`）與標籤（0 = 負面、1 = 正面）。前處理：去掉 `<br />`、轉小寫、切成字（`token`）；字表是 train 裡最常見的 20,000 個字，編號 2～20,001；0 = padding、1 = 沒在字表裡的字（`<unk>`）。每篇只保留**最後** `MAX_LEN` 個字（截斷；第 1 節的 EDA 比較留開頭還是結尾，第 5 節再用 LSTM 驗證）。

**本章最常出現的符號**（各小節開頭有完整符號表）：

| 符號 | 意義 | shape／本章的值 |
| --- | --- | --- |
| $n$、$T$ | batch 大小、batch 裡最長的長度 | 64、≤ `MAX_LEN` |
| $V$、$E$、$H$ | 字表大小（含 0、1）、字向量維度、隱藏維度 | 20,002、64、64 |
| $x_t$ | 第 $t$ 個字的向量 | $(n, E)$ |
| $i_t, f_t, o_t$ | 輸入、遺忘、輸出**閘門**（sigmoid，0～1） | $(n, H)$ |
| $g_t$ | 候選內容（tanh，−1～1） | $(n, H)$ |
| $c_t$ | cell state（長期記憶）；$c_0 = 0$ | $(n, H)$ |
| $h_t$ | 隱藏狀態（輸出）；$h_0 = 0$ | $(n, H)$ |
| $W_x$、$W_h$、$b$ | 四個閘門的參數**並排**：$[i \mid f \mid g \mid o]$ | $(E, 4H)$、$(H, 4H)$、$(4H,)$ |
| $m_t$ | mask（1 = 真的字、0 = padding） | $(n,)$ |

**跨章引用對照**：

| 文中寫法 | 指的是 | 那裡的結論（重述） |
| --- | --- | --- |
| 第 1 課 | MLP＋反向傳播 | 每一層 `forward`／`backward`；`Linear` 的梯度 $X^\top\delta$；softmax＋交叉熵的梯度 $(P - Y)/n$；梯度檢查（中央差分，相對誤差 < 1e-6）。 |
| 第 2 課 | RNN（`03-early-nn/02-rnn`），名字分類 | `Embedding` 用 `np.add.at` 累加梯度；RNN 用 mask 處理 padding（padding 的位置保持舊的 $h$）；BPTT 每往回一步乘 $(1 - h_t^2)W_h^\top$；$W_h = 0.5 \times$ 正交時，梯度傳到 11 步前只剩約 $4 \times 10^{-4}$，$3 \times$ 正交時放大約 86 倍；梯度裁剪限制每一步的位移；在名字資料上 RNN（test macro-F1 0.43）沒贏過字母 n-gram + LogReg（0.56）。 |
| 第一部分第 6 章 | 樸素貝氏（SMS） | 詞袋：只數字出現的次數、不管順序；TF-IDF：常見字（the、a）權重低、少見但有區別力的字權重高。 |
| 第 2 課 3E 實驗 4 | 逐字母的預測 | 每讀一個字就用當下的 $h_t$ 預測一次，看模型什麼時候「確定」。本章用同一招看影評。 |

### 前置複習：本章會用到的舊知識

#### 數學

**1. sigmoid 與它的導數**：$\sigma(z) = 1/(1 + e^{-z})$，值在 0～1，當成「開關」或「比例」。$\sigma'(z) = \sigma(z)(1 - \sigma(z))$。例：$\sigma(0) = 0.5$（導數 0.25）、$\sigma(1) = 0.731$、$\sigma(3) = 0.953$。

**2. tanh**：$\tanh'(a) = 1 - \tanh^2(a)$（第 2 課）。

**3. 逐元素乘法** $\odot$：同 shape 的兩個陣列對應位置相乘（NumPy 的 `*`）。$(f \odot c)_j = f_j c_j$：每一個維度各自的開關。它的導數也是逐元素的：$\partial(f \odot c)/\partial c = f$（不是矩陣乘法）。

**4. 乘積法則**：$\partial(uv) = v\,\partial u + u\,\partial v$。$c_t = f_t \odot c_{t-1} + i_t \odot g_t$ 對 $f_t$ 的導數是 $c_{t-1}$、對 $i_t$ 是 $g_t$、對 $g_t$ 是 $i_t$、對 $c_{t-1}$ 是 $f_t$。

**5. 連乘**（第 2 課）：$0.9^{100} = 2.7 \times 10^{-5}$、$0.99^{100} = 0.37$、$0.999^{100} = 0.90$。影評有上百個字 → 「每一步乘多少」差一點點，傳到開頭就差好幾個數量級。

#### 程式

**6. 把四個矩陣並排成一個**：`z = x @ Wx + h @ Wh + b` 一次算出 $(n, 4H)$，再切成四段：`z[:, :H]`、`z[:, H:2*H]`、`z[:, 2*H:3*H]`、`z[:, 3*H:]`。一次大的矩陣乘法比四次小的快。
**7. `np.concatenate([a, b, c, d], axis=1)`**：把四段梯度接回 $(n, 4H)$。
**8. 正規表示式切字**：`re.findall(r"[a-z0-9']+|[!?.,]", s)` → 連續的英數字（含撇號，`don't` 算一個字）或單一個標點。`re.sub(r"<br\s*/?>", " ", s)` 去掉 HTML 換行。
**9. `Counter.most_common(k)`**：出現次數最多的 $k$ 個（字, 次數）。

---
## 1. 資料與探索性資料分析（EDA）

觀察直接寫在每格程式最上面的註解（`# 觀察：`）。每格附了一兩個引導問題，回答完就可以把問題刪掉。

`load_imdb()` 回傳 Hugging Face 的 `DatasetDict`：`ds["train"]`、`ds["test"]`，每筆有 `text`、`label`（0 = neg、1 = pos）。

In [ ]:
import re
import time
from collections import Counter

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from ml_dl_llm.data import load_imdb

ds = load_imdb()
train_text = [ex["text"] for ex in ds["train"]]
y_full = np.array(ds["train"]["label"])
test_text = [ex["text"] for ex in ds["test"]]
y_test = np.array(ds["test"]["label"])
print("train / test：", len(train_text), len(test_text), "；正面比例：", y_full.mean().round(3), y_test.mean().round(3))
print("\n例（前 400 個字元）：\n", train_text[0][:400])

In [ ]:
# 前處理與長度
# 觀察：
#   - 有多少比例的影評含 <br />？去掉之後會不會黏在一起（看程式：換成空白）？
#   - 字數的中位數、95%、99% 分位數？如果每篇只留 200 個字，有多少比例的影評被截斷？
#   - 正面、負面的長度分布有差別嗎？
def tokenize(s: str) -> list[str]:
    """去掉 <br />、轉小寫，切成字與標點（前置複習 8）。"""
    return re.findall(r"[a-z0-9']+|[!?.,]", re.sub(r"<br\s*/?>", " ", s.lower()))


t0 = time.perf_counter()
train_tok = [tokenize(s) for s in train_text]
test_tok = [tokenize(s) for s in test_text]
print(f"切字花 {time.perf_counter() - t0:.1f} 秒；例：", train_tok[0][:25])
lens = np.array([len(t) for t in train_tok])
print("含 <br /> 的比例：", np.mean(["<br" in s for s in train_text]).round(3))
print("字數分位數（5, 25, 50, 75, 95, 99%）：", np.percentile(lens, [5, 25, 50, 75, 95, 99]).astype(int).tolist(), "；最長：", lens.max())
for L in [100, 200, 400]:
    print(f"   只留 {L} 個字：{np.mean(lens > L):.1%} 的影評被截斷")
fig, ax = plt.subplots(figsize=(8, 3.5))
for lab, name in [(0, "neg"), (1, "pos")]:
    ax.hist(np.minimum(lens[y_full == lab], 1000), bins=80, alpha=0.5, label=name)
ax.set_xlabel("tokens per review (clipped at 1000)")
ax.legend()
plt.show()

In [ ]:
# 字表：Zipf 定律與覆蓋率；最能區分正負面的字
# 觀察：
#   - 不同的字有幾個？只出現 1 次的佔多少？最常見的 20,000 個字涵蓋了多少比例的「字數」？
#   - 「正面／負面出現比例」最懸殊的字是哪些？有沒有你意想不到的？
counts = Counter(w for t in train_tok for w in t)
freq = np.array(sorted(counts.values(), reverse=True))
cum = np.cumsum(freq) / freq.sum()
print(f"不同的字：{len(counts):,}；只出現 1 次：{np.mean(freq == 1):.1%}；前 20,000 個字涵蓋 {cum[19_999]:.2%} 的字數")
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
axes[0].loglog(np.arange(1, len(freq) + 1), freq)
axes[0].set_xlabel("rank")
axes[0].set_ylabel("count")
axes[0].set_title("Zipf")
axes[1].semilogx(np.arange(1, len(freq) + 1), cum)
axes[1].axvline(20_000, color="gray", ls=":")
axes[1].set_xlabel("vocabulary size")
axes[1].set_ylabel("token coverage")
plt.show()

pos_c = Counter(w for t, lab in zip(train_tok, y_full) if lab == 1 for w in set(t))
neg_c = Counter(w for t, lab in zip(train_tok, y_full) if lab == 0 for w in set(t))
rows = [(w, pos_c[w], neg_c[w], np.log((pos_c[w] + 1) / (neg_c[w] + 1))) for w in counts if pos_c[w] + neg_c[w] >= 200]
lr_tbl = pd.DataFrame(rows, columns=["word", "#pos reviews", "#neg reviews", "log ratio"]).sort_values("log ratio")
display(pd.concat([lr_tbl.head(12).reset_index(drop=True), lr_tbl.tail(12).iloc[::-1].reset_index(drop=True)], axis=1, keys=["most negative", "most positive"]))

In [ ]:
# 情緒在影評的哪裡？只看開頭 k 個字 vs 只看結尾 k 個字，用簡單的詞袋 + LogReg 比較
# 觀察：
#   - 同樣 k 個字，開頭和結尾哪個比較能判斷正負面？k = 50 和 k = 200 的差距一樣嗎？
#   - 這對「截斷時留開頭還是結尾」有什麼啟示？（RNN／LSTM 只能看 MAX_LEN 個字）
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

side_rows = []
for k in [50, 200]:
    for side in ["head", "tail"]:
        docs = [" ".join(t[:k] if side == "head" else t[-k:]) for t in train_tok[:10_000]]
        acc = cross_val_score(LogisticRegression(max_iter=2000), CountVectorizer(min_df=3).fit_transform(docs), y_full[:10_000], cv=3).mean()
        side_rows.append({"k": k, "side": side, "3-fold acc (10k reviews)": acc})
display(pd.DataFrame(side_rows).pivot(index="k", columns="side").round(4))

In [ ]:
# TODO 1（問答）EDA 小結
#
# (1) 長度：
#     (a) 字數的中位數與 99% 分位數各是多少？差幾倍？
#     (b) 第 2 課的名字最長 20 個字母。影評長上百個字，第 2 課實驗 1 的梯度衰減（ρ = 0.5 時，11 步就剩 4e-4）放到 200 步會怎樣？
#     (c) 為什麼要截斷（MAX_LEN）？（提示：一個 batch 要補到最長的那篇；BPTT 的時間與記憶體 ∝ T）
# 答：
#
# (2) 字表：前 20,000 個字涵蓋多少比例的字數？剩下沒涵蓋的字會變成 <unk>（編號 1），哪些種類的字最常被丟掉（人名、片名、拼錯字…）？對情感分類影響大嗎？
# 答：
#
# (3) 「最能區分」的字：有沒有不是形容詞、卻很有區分力的字？（例如數字、片名、標點）為什麼？
# 答：
#
# (4) 開頭 vs 結尾：哪一邊資訊多？差距大嗎？本章根據這個結果，預設截斷時「留結尾」—— 你同意嗎？（第 5 節會用 LSTM 再驗證一次）
# 答：

### 資料切分與編碼

- **validation**：從 25,000 篇 train 切 5,000 篇（分層），調參全程只用 train（20,000）／validation；test 到第 6 節才看一次。
- **字表**：只用 train 的 20,000 篇建（validation、test 的字如果不在字表裡就是 `<unk>`）。
- **截斷**：預設 `MAX_LEN = 200`、留**結尾**（依上面的 EDA；第 5 節會比較）。
- `encode(token_lists, max_len, side)` → `Xi` $(n, \text{max\_len})$ 整數、`lengths` $(n,)$。

In [ ]:
from sklearn.model_selection import train_test_split

VOCAB_SIZE = 20_000
MAX_LEN = 200
tr_idx, va_idx = train_test_split(np.arange(len(train_tok)), test_size=5_000, stratify=y_full, random_state=0)
y_tr, y_va = y_full[tr_idx], y_full[va_idx]
vocab_counts = Counter(w for i in tr_idx for w in train_tok[i])
WORD_TO_ID = {w: k + 2 for k, (w, _) in enumerate(vocab_counts.most_common(VOCAB_SIZE))}  # 0 = padding、1 = <unk>
V = VOCAB_SIZE + 2


def encode(token_lists: list[list[str]], max_len: int = MAX_LEN, side: str = "tail") -> tuple[np.ndarray, np.ndarray]:
    """→ (字的編號 (n, max_len)，真正長度 (n,))。side="head" 留開頭、"tail" 留結尾。"""
    Xi = np.zeros((len(token_lists), max_len), dtype=np.int64)
    lengths = np.zeros(len(token_lists), dtype=np.int64)
    for r, toks in enumerate(token_lists):
        ids = [WORD_TO_ID.get(w, 1) for w in toks]
        ids = ids[:max_len] if side == "head" else ids[-max_len:]
        Xi[r, : len(ids)] = ids
        lengths[r] = max(len(ids), 1)  # 空的影評（不會發生，保險）至少長度 1
    return Xi, lengths


tok_tr = [train_tok[i] for i in tr_idx]
tok_va = [train_tok[i] for i in va_idx]
Xi_tr, len_tr = encode(tok_tr)
Xi_va, len_va = encode(tok_va)
Xi_te, len_te = encode(test_tok)
print("Xi_tr", Xi_tr.shape, "；validation 的 <unk> 比例：", f"{np.mean(Xi_va[Xi_va > 0] == 1):.2%}", "；test：", f"{np.mean(Xi_te[Xi_te > 0] == 1):.2%}")

---
## 2. Baseline

| Baseline | 做法 | 看什麼 |
| --- | --- | --- |
| Dummy（prior） | 永遠輸出 0.5／0.5 | 最低標準（類別平衡 → accuracy 0.5） |
| 詞袋 + LogReg | 每個字出現的次數（`CountVectorizer`），**整篇**、不截斷 | 「不管順序」能做到多少？ |
| TF-IDF（1～2-gram）+ LogReg | 字與相鄰兩個字（`not good`）、TF-IDF 加權 | 經典的強 baseline；LSTM 要打敗的對手 |

注意：詞袋模型看的是**整篇**影評；第 3～5 節的 RNN／LSTM 只看最後 `MAX_LEN` 個字。

**評估指標**（validation，之後一路往下加到 `val_table`）：accuracy、log loss、ROC AUC。

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from sklearn.pipeline import make_pipeline


def val_scores(prob: np.ndarray, y: np.ndarray = y_va) -> dict[str, float]:
    """prob (n, 2) 是預測機率。回傳 {"accuracy", "log_loss", "roc_auc"}。"""
    return {
        "accuracy": round(accuracy_score(y, prob.argmax(axis=1)), 4),
        "log_loss": round(log_loss(y, np.clip(prob, 1e-12, 1), labels=[0, 1]), 4),
        "roc_auc": round(roc_auc_score(y, prob[:, 1]), 4),
    }


docs_tr = [" ".join(train_tok[i]) for i in tr_idx]
docs_va = [" ".join(train_tok[i]) for i in va_idx]
val_table = pd.DataFrame(columns=["accuracy", "log_loss", "roc_auc"])
val_table.loc["Dummy prior"] = val_scores(DummyClassifier(strategy="prior").fit(np.zeros((len(y_tr), 1)), y_tr).predict_proba(np.zeros((len(y_va), 1))))
t0 = time.perf_counter()
bow = make_pipeline(CountVectorizer(min_df=2, max_features=VOCAB_SIZE), LogisticRegression(max_iter=3000)).fit(docs_tr, y_tr)
val_table.loc["BoW (20k words) + LogReg"] = val_scores(bow.predict_proba(docs_va))
tfidf = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200_000, token_pattern=r"\S+"), LogisticRegression(C=4, max_iter=3000)).fit(docs_tr, y_tr)
val_table.loc["TF-IDF 1-2-gram + LogReg"] = val_scores(tfidf.predict_proba(docs_va))
print(f"兩個 LogReg 共 {time.perf_counter() - t0:.0f} 秒")
docs_va_cut = [" ".join(train_tok[i][-MAX_LEN:]) for i in va_idx]
val_table.loc[f"TF-IDF 1-2-gram + LogReg (val: last {MAX_LEN} tokens only)"] = val_scores(tfidf.predict_proba(docs_va_cut))
display(val_table)

In [ ]:
# TODO 2（問答）Baseline
#
# (1) 讀 val_table：BoW、TF-IDF 的 accuracy 各是多少？加上 2-gram 與 TF-IDF 加權進步多少？
#     2-gram 能抓到哪些 1-gram 抓不到的東西？（舉例：not good、not bad、waste of）
# 答：
#
# (2) 同一個 TF-IDF 模型，validation 只給最後 200 個字時掉多少？這代表 RNN／LSTM「只看 200 個字」先天少了多少資訊？
# 答：
#
# (3) 先寫下預測（第 4、6 節驗證）：
#     (a) 第 2 課的 RNN 放到 200 個字的影評上，accuracy 大約多少？（提示：TODO 1 (1)(b)）
#     (b) LSTM 會贏過 TF-IDF + LogReg 嗎？
# 答：

---
## 3. 手寫模型（NumPy）

| 小節 | 做什麼 | TODO |
| --- | --- | --- |
| 3A | `LSTM` 的 forward：四個閘門、cell state、mask | 3、4 |
| 3B | LSTM 的反向傳播：為什麼 $c$ 那條路不會消失 | 5（推導）、6 |
| 3C | 組成 `TextRNNNumpy`（可換 RNN／LSTM）、梯度檢查、在**子集**上訓練 | 7 |
| 3D | 實驗：梯度能傳多遠、遺忘閘門偏差、閘門在做什麼、逐字的預測 | 8 |

**NumPy 版只在子集上訓練**：LSTM 每一步要算 $4H$ 個值，NumPy 版一個 batch（64 篇 × 200 個字）約 1～2 秒，跑完整個訓練集一個 epoch 要好幾分鐘。所以 3C 用 **8,000 篇、每篇最後 100 個字、$E = 32$、$H = 64$** 驗證演算法、做實驗；完整的訓練交給第 4 節的 PyTorch。

### 從第 1、2 課搬過來的層（已完成）

- 第 1 課：`Linear`、`ReLU`、`Tanh`、`SoftmaxCrossEntropy`、`init_std`、`numerical_grad`、`rel_error`、`check_layer`；
- 第 2 課：`Embedding`（`np.add.at`）、`RNN`（含 mask、BPTT、`dh_norms`）、`clip_grad_norm`。

都是原本寫好的版本，原封不動。

In [ ]:
from typing import Literal, Self
from numpy.typing import ArrayLike
from scipy.special import logsumexp

InitName = Literal["he", "xavier"]


def init_std(init: InitName | float, fan_in: int, fan_out: int) -> float:
    """權重初始化的標準差。He：sqrt(2 / fan_in)（給 ReLU）；Xavier：sqrt(2 / (fan_in + fan_out))（給 tanh）。"""
    if isinstance(init, (int, float)):
        return float(init)
    if init == "he":
        return np.sqrt(2 / fan_in)
    if init == "xavier":
        return np.sqrt(2 / (fan_in + fan_out))
    raise ValueError(init)


class Layer:
    """所有層的共同介面。沒有參數的層，params／grads 是空 dict。"""

    def __init__(self) -> None:
        self.params: dict[str, np.ndarray] = {}
        self.grads: dict[str, np.ndarray] = {}

    def forward(self, X: np.ndarray) -> np.ndarray:
        raise NotImplementedError

    def backward(self, dout: np.ndarray) -> np.ndarray:
        raise NotImplementedError


class Linear(Layer):
    """全連接層：out = X @ W + b。X (n, d_in)、W (d_in, d_out)、b (d_out,)。"""

    def __init__(
        self,
        d_in: int,
        d_out: int,
        init: InitName | float = "he",
        rng: np.random.Generator | None = None,
        dtype: type = np.float64,
    ) -> None:
        super().__init__()
        rng = np.random.default_rng() if rng is None else rng
        W = rng.normal(0.0, init_std(init, d_in, d_out), size=(d_in, d_out))
        self.params = {"W": W.astype(dtype), "b": np.zeros(d_out, dtype=dtype)}
        self.grads = {k: np.zeros_like(v) for k, v in self.params.items()}

    def forward(self, X: np.ndarray) -> np.ndarray:
        self.X = X
        return X @ self.params["W"] + self.params["b"]

    def backward(self, dout: np.ndarray) -> np.ndarray:
        self.grads["W"] = self.X.T @ dout
        self.grads["b"] = dout.sum(axis=0)
        return dout @ self.params["W"].T


class ReLU(Layer):
    """out = max(0, X)。"""

    def forward(self, X: np.ndarray) -> np.ndarray:
        self.mask = X > 0
        return X * self.mask

    def backward(self, dout: np.ndarray) -> np.ndarray:
        return dout * self.mask


class Tanh(Layer):
    """out = tanh(X)。"""

    def forward(self, X: np.ndarray) -> np.ndarray:
        self.out = np.tanh(X)
        return self.out

    def backward(self, dout: np.ndarray) -> np.ndarray:
        return dout * (1 - self.out**2)


class SoftmaxCrossEntropy:
    """softmax + 平均交叉熵。forward(Z, y) 回傳純量 loss；backward() 回傳 ∂L/∂Z = (P − Y)/n。"""

    def forward(self, Z: np.ndarray, y: np.ndarray) -> float:
        log_P = Z - logsumexp(Z, axis=1, keepdims=True)
        self.P = np.exp(log_P)
        self.y = y
        return float(-log_P[np.arange(len(y)), y].mean())

    def backward(self) -> np.ndarray:
        n = len(self.y)
        dZ = self.P.copy()
        dZ[np.arange(n), self.y] -= 1
        return dZ / n


def numerical_grad(f, x: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """中央差分：回傳與 x 同 shape 的 ∂f/∂x。會就地修改 x 再改回來。"""
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=["multi_index"])
    for _ in it:
        i = it.multi_index
        old = x[i]
        x[i] = old + eps
        f_plus = f()
        x[i] = old - eps
        f_minus = f()
        x[i] = old
        grad[i] = (f_plus - f_minus) / (2 * eps)
    return grad


def rel_error(a: np.ndarray, b: np.ndarray) -> float:
    """相對誤差 ‖a − b‖ / ‖a + b‖（兩者都是 0 時回傳 0）。"""
    denom = np.linalg.norm(a + b)
    return 0.0 if denom == 0 else float(np.linalg.norm(a - b) / denom)


def check_layer(layer: Layer, X: np.ndarray, seed: int = 0) -> dict[str, float]:
    """單一層的梯度檢查（float64）。f(X) = Σ forward(X) ⊙ G，G 是隨機的上游梯度。

    回傳 {"dX": 相對誤差, "dW": ..., "db": ...}（沒有參數的層只有 dX）。
    """
    G = np.random.default_rng(seed).normal(size=layer.forward(X).shape)
    f = lambda: float(np.sum(layer.forward(X) * G))
    layer.forward(X)
    dX = layer.backward(G)
    errs = {"dX": rel_error(dX, numerical_grad(f, X))}
    for k in layer.params:
        errs["d" + k] = rel_error(layer.grads[k], numerical_grad(f, layer.params[k]))
    return errs

In [ ]:
class Embedding(Layer):
    """查表：out = W[X]。X (n, T) 是整數（0 = padding，那一列固定是 0）、W (V, E)、out (n, T, E)。"""

    def __init__(self, V: int, E: int, std: float = 0.1, rng: np.random.Generator | None = None) -> None:
        super().__init__()
        rng = np.random.default_rng() if rng is None else rng
        W = rng.normal(0.0, std, size=(V, E))
        W[0] = 0.0
        self.params = {"W": W}
        self.grads = {"W": np.zeros_like(W)}

    def forward(self, X: np.ndarray) -> np.ndarray:
        self.X = X
        return self.params["W"][X]

    def backward(self, dout: np.ndarray) -> None:
        """回傳 None：輸入是整數編號，沒有「對輸入的梯度」。"""
        dW = np.zeros_like(self.params["W"])
        np.add.at(dW, self.X, dout)
        dW[0] = 0.0  # padding 的向量不更新
        self.grads["W"] = dW
        return None


class RNN(Layer):
    """單層 tanh RNN（batch_first）。forward(X, mask) 回傳最後的隱藏狀態 (n, H)。

    forward 存下的 cache：self.X (n, T, E)、self.mask (n, T)、self.Hs (n, T + 1, H)。
    backward 另外存 self.dh_norms (n, T)：‖∂L/∂h‖（讀完第 t 個字母後的隱藏狀態），實驗用。
    """

    def __init__(self, E: int, H: int, rng: np.random.Generator | None = None) -> None:
        super().__init__()
        rng = np.random.default_rng() if rng is None else rng
        self.params = {
            "Wx": rng.normal(0.0, np.sqrt(1 / E), size=(E, H)),
            "Wh": rng.normal(0.0, np.sqrt(1 / H), size=(H, H)),
            "b": np.zeros(H),
        }
        self.grads = {k: np.zeros_like(v) for k, v in self.params.items()}

    def forward(self, X: np.ndarray, mask: np.ndarray) -> np.ndarray:
        n, T, _ = X.shape
        Wx, Wh, b = self.params["Wx"], self.params["Wh"], self.params["b"]
        Hs = np.zeros((n, T + 1, Wh.shape[0]))
        for t in range(T):
            h_new = np.tanh(X[:, t] @ Wx + Hs[:, t] @ Wh + b)
            m = mask[:, t, None]
            Hs[:, t + 1] = m * h_new + (1 - m) * Hs[:, t]
        self.X, self.mask, self.Hs = X, mask, Hs
        return Hs[:, -1]

    def backward(self, dh_last: np.ndarray) -> np.ndarray:
        """dh_last = ∂L/∂h_T (n, H)。回傳 dX (n, T, E)，並把參數的梯度存進 self.grads。"""
        X, mask, Hs = self.X, self.mask, self.Hs
        n, T, _ = X.shape
        Wx, Wh = self.params["Wx"], self.params["Wh"]
        self.grads = {k: np.zeros_like(v) for k, v in self.params.items()}
        dX = np.zeros_like(X)
        self.dh_norms = np.zeros((n, T))
        dh = dh_last
        for t in range(T - 1, -1, -1):
            self.dh_norms[:, t] = np.linalg.norm(dh, axis=1)
            m = mask[:, t, None]
            h, h_prev = Hs[:, t + 1], Hs[:, t]
            da = dh * m * (1 - h**2)
            self.grads["Wx"] += X[:, t].T @ da
            self.grads["Wh"] += h_prev.T @ da
            self.grads["b"] += da.sum(axis=0)
            dX[:, t] = da @ Wx.T
            dh = da @ Wh.T + dh * (1 - m)
        return dX


def clip_grad_norm(grads: list[np.ndarray], max_norm: float) -> float:
    """就地把梯度縮放到全域範數 ≤ max_norm。回傳縮放前的全域範數。"""
    total = float(np.sqrt(sum((g**2).sum() for g in grads)))
    scale = 1.0 if total == 0 else min(1.0, max_norm / total)
    for g in grads:
        g *= scale
    return total

### 3A. `LSTM` 的 forward

每一步（$t = 1, \dots, T$）：

$$
\begin{aligned}
z_t &= x_t W_x + h_{t-1} W_h + b \quad (n, 4H)\ \text{，切成四段} \\
i_t &= \sigma(z_t^{(i)}) && \text{輸入閘門：這一步的新內容要寫入多少} \\
f_t &= \sigma(z_t^{(f)}) && \text{遺忘閘門：舊的記憶留多少（1 = 全留、0 = 全擦）} \\
g_t &= \tanh(z_t^{(g)}) && \text{候選內容：要寫入的東西（−1～1）} \\
o_t &= \sigma(z_t^{(o)}) && \text{輸出閘門：記憶要露出多少給 } h_t \\
c_t &= f_t \odot c_{t-1} + i_t \odot g_t && \text{cell state：擦掉一部分舊的、加上一部分新的} \\
h_t &= o_t \odot \tanh(c_t)
\end{aligned}
$$

跟第 2 課 RNN 比：RNN 的 $h_t = \tanh(\cdots)$ **每一步都整個重算**；LSTM 的 $c_t$ 是**加法**更新 —— 如果 $f_t = 1$、$i_t = 0$，$c_t = c_{t-1}$，資訊原封不動地往後傳。

**mask**：跟第 2 課一樣，padding 的位置 $c$、$h$ **都**保持舊的值。

**遺忘閘門偏差初始化成 1**（Gers et al. 2000；Jozefowicz et al. 2015）：$b^{(f)} = 1$ → 一開始 $f_t \approx \sigma(1) = 0.73$，偏向「記住」。第 3D 節實驗 2 比較 0 與 1。

#### 符號表（3A）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $X$ | 字向量（Embedding 的輸出） | $(n, T, E)$ | `X` |
| $W_x$、$W_h$、$b$ | 四個閘門並排 $[i \mid f \mid g \mid o]$ | $(E, 4H)$、$(H, 4H)$、$(4H,)$ | `params["Wx"]`、`params["Wh"]`、`params["b"]` |
| $XW_x$ | 所有時間步的 $x_t W_x$ 一次先算好 | $(n, T, 4H)$ | `XW` |
| $z_t$ | 四段並排的「閘門前」值 | $(n, 4H)$ | `z` |
| $i, f, g, o$ | 四個閘門／候選內容 | 各 $(n, H)$ | `i`、`f`、`g`、`o` |
| Cs、Hs | $c_0..c_T$、$h_0..h_T$ | $(n, T + 1, H)$ | `self.Cs`、`self.Hs` |
| Gates | 每一步的 $[i, f, g, o]$（反向傳播要用） | $(n, T, 4H)$ | `self.Gates` |

**下標公式**：
- TODO 3：`i = sigmoid(z[:, :H])`、`f = sigmoid(z[:, H:2*H])`、`g = np.tanh(z[:, 2*H:3*H])`、`o = sigmoid(z[:, 3*H:])`（前置複習 6）。
- TODO 4：`c = f * Cs[:, t] + i * g`；`h = o * np.tanh(c)`；再用 mask：`Cs[:, t + 1] = m * c + (1 - m) * Cs[:, t]`，`Hs` 同理。

#### 最小例子：TODO 3、4（$E = H = 1$ 的 LSTM，一步）

$x_1 = 1$、$h_0 = c_0 = 0$；$W_x = [0.5, 1.0, 2.0, -1.0]$（$i, f, g, o$ 各一個數字）、$W_h = 0$、$b = 0$。

1. **正確答案**（純量，一個閘門一個閘門算）：$z = (0.5, 1.0, 2.0, -1.0)$ → $i = \sigma(0.5) = 0.6225$、$f = \sigma(1) = 0.7311$、$g = \tanh(2) = 0.9640$、$o = \sigma(-1) = 0.2689$；
   $c_1 = 0.7311 \times 0 + 0.6225 \times 0.9640 = 0.6001$；$h_1 = 0.2689 \times \tanh(0.6001) = 0.2689 \times 0.5370 = 0.1444$。
2. **積木**：$z$ 的切片 —— $H = 1$ 時 `z[:, :1]`、`z[:, 1:2]`…
3. **留給你**：寫進 `LSTM.forward`。

In [ ]:
# 最小例子：TODO 3、4（E = H = 1 的 LSTM，一步）
def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1 / (1 + np.exp(-z))


z_list = [0.5, 1.0, 2.0, -1.0]
# ---- 1. 正確答案：純量 ----
i1, f1, g1, o1 = sigmoid(z_list[0]), sigmoid(z_list[1]), np.tanh(z_list[2]), sigmoid(z_list[3])
c1 = f1 * 0.0 + i1 * g1
h1 = o1 * np.tanh(c1)
print(f"i = {i1:.4f}、f = {f1:.4f}、g = {g1:.4f}、o = {o1:.4f}；c_1 = {c1:.4f}、h_1 = {h1:.4f}")

# ---- 2. 積木：矩陣版的 z 與切片 ----
x_mini = np.array([[[1.0]]])  # (n=1, T=1, E=1)
Wx_mini = np.array([[0.5, 1.0, 2.0, -1.0]])  # (E=1, 4H=4)
z_mini = x_mini[:, 0] @ Wx_mini
H_ = 1
print("z =", z_mini, "；z[:, :H] =", z_mini[:, :H_], "；z[:, H:2H] =", z_mini[:, H_ : 2 * H_], "；z[:, 2H:3H] =", z_mini[:, 2 * H_ : 3 * H_], "；z[:, 3H:] =", z_mini[:, 3 * H_ :])

# ---- 3. 留給你 ----
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照（應為 c = 0.6001、h = 0.1444）：
# lstm_mini = LSTM(1, 1, forget_bias=0.0); lstm_mini.params.update(Wx=Wx_mini, Wh=np.zeros((1, 4))); print(lstm_mini.forward(x_mini, np.ones((1, 1))).round(4), lstm_mini.Cs[:, 1].round(4))

In [ ]:
class LSTM(Layer):
    """單層 LSTM（batch_first）。forward(X, mask) 回傳最後的隱藏狀態 h_T (n, H)。

    參數的四段順序 [i | f | g | o]（跟 PyTorch 一樣）。
    forward 存下的 cache：self.X、self.mask、self.Hs、self.Cs (n, T + 1, H)、self.Gates (n, T, 4H)。
    backward 另外存 self.dh_norms、self.dc_norms (n, T)：‖∂L/∂h_t‖、‖∂L/∂c_t‖，實驗用。
    """

    def __init__(self, E: int, H: int, forget_bias: float = 1.0, rng: np.random.Generator | None = None) -> None:
        super().__init__()
        rng = np.random.default_rng() if rng is None else rng
        b = np.zeros(4 * H)
        b[H : 2 * H] = forget_bias
        self.params = {
            "Wx": rng.normal(0.0, np.sqrt(1 / E), size=(E, 4 * H)),
            "Wh": rng.normal(0.0, np.sqrt(1 / H), size=(H, 4 * H)),
            "b": b,
        }
        self.grads = {k: np.zeros_like(v) for k, v in self.params.items()}
        self.H = H

    def forward(self, X: np.ndarray, mask: np.ndarray) -> np.ndarray:
        n, T, _ = X.shape
        H, Wh, b = self.H, self.params["Wh"], self.params["b"]
        XW = X @ self.params["Wx"]  # (n, T, 4H)：所有時間步的 x_t Wx 一次算好
        Hs = np.zeros((n, T + 1, H))
        Cs = np.zeros((n, T + 1, H))
        Gates = np.zeros((n, T, 4 * H))
        for t in range(T):
            z = XW[:, t] + Hs[:, t] @ Wh + b
            # TODO 3（程式）：四個閘門（見下標公式）
            raise NotImplementedError("TODO 3：四個閘門")
            # TODO 4（程式）：c、h，再用 mask 決定 Cs[:, t + 1]、Hs[:, t + 1]
            raise NotImplementedError("TODO 4：cell state 與 h")
            Gates[:, t] = np.concatenate([i, f, g, o], axis=1)
        self.X, self.mask, self.Hs, self.Cs, self.Gates = X, mask, Hs, Cs, Gates
        return Hs[:, -1]

    def backward(self, dh_last: np.ndarray) -> np.ndarray:
        """dh_last = ∂L/∂h_T (n, H)。回傳 dX (n, T, E)，參數的梯度存進 self.grads。"""
        X, mask, Hs, Cs, Gates = self.X, self.mask, self.Hs, self.Cs, self.Gates
        n, T, _ = X.shape
        H = self.H
        dZ = np.zeros((n, T, 4 * H))
        dh, dc = dh_last, np.zeros_like(dh_last)
        dWh = np.zeros_like(self.params["Wh"])
        self.dh_norms, self.dc_norms = np.zeros((n, T)), np.zeros((n, T))
        for t in range(T - 1, -1, -1):
            self.dh_norms[:, t], self.dc_norms[:, t] = np.linalg.norm(dh, axis=1), np.linalg.norm(dc, axis=1)
            m = mask[:, t, None]
            i, f, g, o = Gates[:, t, :H], Gates[:, t, H : 2 * H], Gates[:, t, 2 * H : 3 * H], Gates[:, t, 3 * H :]
            c_prev = Cs[:, t]
            tanh_c = np.tanh(f * c_prev + i * g)  # 這一步「真的」c 的 tanh（padding 的列不會用到）
            # TODO 6（程式）：LSTM 的 BPTT 一步（見 3B 的表格，由上往下照抄）
            #   - dh_m = dh * m；dc_m = dc * m（padding 的位置不流進閘門）
            #   - dc_tot = dc_m + dh_m * o * (1 − tanh_c²)
            #   - dz = np.concatenate([dc_tot * g * i * (1 − i),          ← z 的 i 段
            #                          dc_tot * c_prev * f * (1 − f),     ← f 段
            #                          dc_tot * i * (1 − g²),             ← g 段
            #                          dh_m * tanh_c * o * (1 − o)], axis=1)  ← o 段
            #   - dZ[:, t] = dz；dWh += Hs[:, t].T @ dz
            #   - dh = dz @ Wh.T + dh * (1 − m)；dc = dc_tot * f + dc * (1 − m)
            #   常見錯誤：dc 往回傳時乘的是 f（逐元素），不是矩陣；o 段用的是 tanh_c 而不是 c
            raise NotImplementedError("TODO 6：LSTM 的 BPTT")
        # Wx、b 的梯度：所有時間步一起算（跟 forward 的 XW 一樣，一次大的矩陣乘法）
        self.grads = {"Wx": np.einsum("nte,ntk->ek", X, dZ), "Wh": dWh, "b": dZ.sum(axis=(0, 1))}
        return dZ @ self.params["Wx"].T

### 3B. LSTM 的反向傳播

從 $\delta^h_t = \partial L/\partial h_t$ 和 $\delta^c_t = \partial L/\partial c_t$（**兩條**往回傳的梯度）出發，每往回一步（先假設不是 padding）：

| 量 | 公式 | 怎麼來的 |
| --- | --- | --- |
| $\partial L/\partial c_t$（總） | $\delta^c_t + \delta^h_t \odot o_t \odot (1 - \tanh^2 c_t)$ | $c_t$ 影響 $L$ 兩條路：直接傳給 $c_{t+1}$（$\delta^c_t$），以及透過 $h_t = o_t\tanh(c_t)$ |
| $\partial L/\partial o_t$ | $\delta^h_t \odot \tanh(c_t)$ | $h_t = o_t \odot \tanh c_t$ |
| $\partial L/\partial i_t$ | $\partial L/\partial c_t \odot g_t$ | $c_t = f_t c_{t-1} + i_t g_t$（前置複習 4） |
| $\partial L/\partial g_t$ | $\partial L/\partial c_t \odot i_t$ | 同上 |
| $\partial L/\partial f_t$ | $\partial L/\partial c_t \odot c_{t-1}$ | 同上 |
| $\partial L/\partial z$ 的四段 | 上面四個各乘上啟動函數的導數：$i(1 - i)$、$f(1 - f)$、$1 - g^2$、$o(1 - o)$ | 前置複習 1、2 |
| $\partial L/\partial W_x, W_h, b$ | 跟 RNN 一樣：$x_t^\top dz$、$h_{t-1}^\top dz$、$\sum dz$，**每一步加總** | 參數共用 |
| $\delta^h_{t-1}$ | $dz\, W_h^\top$ | $h_{t-1}$ 只透過 $z_t$ 影響 $L$ |
| $\delta^c_{t-1}$ | $\partial L/\partial c_t \odot f_t$ | **只乘遺忘閘門**（逐元素），沒有矩陣、沒有 tanh 的導數 |

**padding**（mask = 0）：跟第 2 課一樣，$c_t = c_{t-1}$、$h_t = h_{t-1}$ → 梯度原封不動往回傳，不流進閘門。

最後一行就是 LSTM 的關鍵：沿著 $c$ 往回 $k$ 步，梯度乘上 $f_t \odot f_{t-1} \odot \cdots$。只要遺忘閘門接近 1，梯度就幾乎不衰減 —— 原論文叫它 **constant error carousel**（恆定誤差的旋轉木馬）。

#### 符號表（3B）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $\delta^h_t$、$\delta^c_t$ | 往回傳的兩條梯度 | $(n, H)$ | `dh`、`dc` |
| $\partial L/\partial c_t$（總） | 兩條路加起來 | $(n, H)$ | `dc_tot` |
| $dz$ | 四段並排的「閘門前」梯度 | $(n, 4H)$ | `dz`（全部存進 `dZ`） |
| $c_{t-1}$、$\tanh c_t$ | forward 的 cache | $(n, H)$ | `c_prev`、`tanh_c` |

In [ ]:
# TODO 5（問答）推導與比較（先在紙上，再寫 TODO 6）
#
# (1) 一步的 LSTM（E = H = 1），loss L = h_1，用 TODO 3 最小例子的數字（c_0 = 0）：
#     (a) ∂L/∂o = tanh(c_1) = ?；∂L/∂c_1 = o · (1 − tanh² c_1) = ?
#     (b) ∂L/∂f = ∂L/∂c_1 · c_0 = ? —— 為什麼第一步的遺忘閘門學不到東西？
#     (c) 下一格的最小例子會用數值微分驗證 ∂L/∂Wx（四個數字）。
# 答：
#
# (2) 沿著 c 往回傳 k 步：δ^c 乘上 f_t ⊙ f_{t−1} ⊙ ⋯。
#     (a) 如果每個 f 都是 0.73（σ(1)，初始化時的值），100 步之後剩多少？如果每個 f 都是 0.99 呢？（前置複習 5）
#     (b) 第 2 課 RNN 往回一步乘的是 (1 − h²) · Wh（矩陣）。哪一個比較容易「剛好接近 1」？為什麼 LSTM 可以讓某些維度的 f 接近 1、其他維度接近 0？
# 答：
#
# (3) LSTM 的梯度一定不會消失嗎？（提示：δ^h 那條路還是要經過 Wh；如果模型學到 f ≈ 0 呢？）
# 答：

#### 最小例子：TODO 6（LSTM 一步的反向傳播）

同 TODO 3 的最小例子，loss $= h_1$（$\delta^h_1 = 1$、$\delta^c_1 = 0$）。

1. **正確答案**：數值微分 ∂L/∂Wx（四個數字，對應 $i, f, g, o$）。應為 $\approx (0.0434, 0, 0.0084, 0.1056)$ —— $f$ 那一個是 0（因為 $c_0 = 0$，TODO 5 (1)(b)）。
2. **積木**：$\partial L/\partial c_1$ 與 $\partial L/\partial o$ 先算出來。
3. **留給你**：TODO 6 的四段 $dz$，乘上 $x_1 = 1$ 就是 ∂L/∂Wx。

In [ ]:
# 最小例子：TODO 6（LSTM 一步的反向傳播）
def loss_one_step(Wx_row: np.ndarray) -> float:
    i_, f_, g_, o_ = sigmoid(Wx_row[0]), sigmoid(Wx_row[1]), np.tanh(Wx_row[2]), sigmoid(Wx_row[3])
    return o_ * np.tanh(f_ * 0.0 + i_ * g_)


# ---- 1. 正確答案：數值微分 ----
w0 = np.array([0.5, 1.0, 2.0, -1.0])
num_dWx = np.array([(loss_one_step(w0 + e) - loss_one_step(w0 - e)) / 2e-6 for e in np.eye(4) * 1e-6])
print("數值微分 ∂L/∂Wx（i, f, g, o）：", num_dWx.round(4))

# ---- 2. 積木 ----
dc1 = o1 * (1 - np.tanh(c1) ** 2)
print(f"∂L/∂o = tanh(c_1) = {np.tanh(c1):.4f}；∂L/∂c_1 = o·(1 − tanh² c_1) = {dc1:.4f}；o 段：∂L/∂o · o(1 − o) = {np.tanh(c1) * o1 * (1 - o1):.4f}")

# ---- 3. 留給你 ----
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照（應跟數值微分一樣）：
# lstm_mini = LSTM(1, 1, forget_bias=0.0); lstm_mini.params.update(Wx=Wx_mini.copy(), Wh=np.zeros((1, 4))); lstm_mini.forward(x_mini, np.ones((1, 1))); lstm_mini.backward(np.ones((1, 1))); print(lstm_mini.grads["Wx"].round(4))

### 3C. 組成 `TextRNNNumpy`、梯度檢查、訓練

**程式結構圖**：

```
TextRNNNumpy(cell="lstm" 或 "rnn")
├── emb  : Embedding                      params {"W": (V, E)}
├── rnn  : LSTM（或第 2 課的 RNN）           params {"Wx", "Wh", "b"}     cache: Hs、(Cs、Gates)
├── fc   : Linear                         params {"W": (H, 2), "b"}
├── loss : SoftmaxCrossEntropy（2 類）
└── layers = [emb, rnn, fc]
```

跟第 2 課的 `CharRNNNumpy` 幾乎一樣，只是多了 `cell` 參數、輸入是 `(Xi, lengths)`（mask 在裡面用 `lengths` 產生）。`fit`：mini-batch + 梯度裁剪；batch 依長度排序後切；每個 epoch 量 validation；最後換回 validation loss 最低的權重。

**最佳化器換成 Adam**（已寫好的 `adam_step`）：在這份資料上，同樣的 epoch 數 SGD 幾乎學不動（試過 lr = 0.5、1.0：子集上 6 個 epoch 只到 0.5～0.64）。Adam 對每個參數各自調整步長：
$$m \leftarrow \beta_1 m + (1 - \beta_1)g,\quad v \leftarrow \beta_2 v + (1 - \beta_2)g^2,\quad \theta \leftarrow \theta - \eta\,\frac{\hat m}{\sqrt{\hat v} + \epsilon}$$
（$\hat m, \hat v$ 是除以 $1 - \beta^t$ 的偏差修正。）細節放在待補強；這裡先把它當成「比 SGD 好調的 SGD」，跟第 2 課 PyTorch 用的是同一個演算法。

**子集**：train 前 8,000 篇、validation 前 1,000 篇，每篇最後 100 個字，$E = 32$、$H = 64$，Adam lr = 2e-3，6 epochs。

In [ ]:
def lengths_to_mask(lengths: np.ndarray, T: int) -> np.ndarray:
    """(n,) → (n, T) 的 0/1 mask。"""
    return (np.arange(T)[None, :] < lengths[:, None]).astype(float)


class TextRNNNumpy:
    """Embedding → LSTM（或 RNN）→ Linear → softmax（2 類）。"""

    def __init__(self, V: int, cell: str = "lstm", E: int = 32, H: int = 64, lr: float = 2e-3, batch_size: int = 64,
                 n_epochs: int = 6, clip: float = 5.0, forget_bias: float = 1.0, seed: int = 0) -> None:
        rng = np.random.default_rng(seed)
        self.emb = Embedding(V, E, rng=rng)
        self.rnn = LSTM(E, H, forget_bias=forget_bias, rng=rng) if cell == "lstm" else RNN(E, H, rng=rng)
        self.fc = Linear(H, 2, init="xavier", rng=rng)
        self.loss = SoftmaxCrossEntropy()
        self.layers = [self.emb, self.rnn, self.fc]
        self.cell, self.lr, self.batch_size, self.n_epochs, self.clip, self.seed = cell, lr, batch_size, n_epochs, clip, seed

    def _param_list(self) -> list[tuple[dict, dict, str]]:
        return [(layer.params, layer.grads, k) for layer in self.layers for k in layer.params]

    def _adam_step(self, beta1: float = 0.9, beta2: float = 0.999, eps: float = 1e-8) -> None:
        """Adam（已寫好）：每個參數有自己的 m（梯度的移動平均）、v（梯度平方的移動平均）。"""
        if not hasattr(self, "_adam_t"):
            self._adam_t = 0
            self._adam_m = [np.zeros_like(p[k]) for p, _, k in self._param_list()]
            self._adam_v = [np.zeros_like(p[k]) for p, _, k in self._param_list()]
        self._adam_t += 1
        for (p, g, k), m, v in zip(self._param_list(), self._adam_m, self._adam_v):
            m *= beta1
            m += (1 - beta1) * g[k]
            v *= beta2
            v += (1 - beta2) * g[k] ** 2
            m_hat = m / (1 - beta1**self._adam_t)
            v_hat = v / (1 - beta2**self._adam_t)
            p[k] -= self.lr * m_hat / (np.sqrt(v_hat) + eps)

    def forward(self, Xi: np.ndarray, lengths: np.ndarray) -> np.ndarray:
        T = int(lengths.max())
        return self.fc.forward(self.rnn.forward(self.emb.forward(Xi[:, :T]), lengths_to_mask(lengths, T)))

    def loss_and_backward(self, Xi: np.ndarray, lengths: np.ndarray, y: np.ndarray) -> float:
        loss = self.loss.forward(self.forward(Xi, lengths), y)
        self.emb.backward(self.rnn.backward(self.fc.backward(self.loss.backward())))
        return loss

    def predict_proba(self, Xi: np.ndarray, lengths: np.ndarray, batch_size: int = 500) -> np.ndarray:
        out = []
        for s in range(0, len(Xi), batch_size):
            Z = self.forward(Xi[s : s + batch_size], lengths[s : s + batch_size])
            P = np.exp(Z - Z.max(axis=1, keepdims=True))
            out.append(P / P.sum(axis=1, keepdims=True))
        return np.vstack(out)

    def fit(self, Xi, lengths, y, Xi_val=None, len_val=None, y_val=None, verbose: bool = True) -> "TextRNNNumpy":
        rng = np.random.default_rng(self.seed + 1)
        order = np.argsort(lengths, kind="stable")
        batches = [order[s : s + self.batch_size] for s in range(0, len(order), self.batch_size)]
        self.history_ = {"train_loss": [], "val_loss": [], "val_acc": [], "seconds": []}
        best_loss, best_state = np.inf, None
        for epoch in range(self.n_epochs):
            t0, total = time.perf_counter(), 0.0
            for b in rng.permutation(len(batches)):
                idx = batches[b]
                total += self.loss_and_backward(Xi[idx], lengths[idx], y[idx]) * len(idx)
                clip_grad_norm([g[k] for _, g, k in self._param_list()], self.clip)
                self._adam_step()
            self.history_["train_loss"].append(total / len(y))
            msg = f"epoch {epoch + 1:2d}  train_loss {total / len(y):.4f}"
            if Xi_val is not None:
                s_ = val_scores(self.predict_proba(Xi_val, len_val), y_val)
                self.history_["val_loss"].append(s_["log_loss"])
                self.history_["val_acc"].append(s_["accuracy"])
                msg += f"  val_loss {s_['log_loss']:.4f}  val_acc {s_['accuracy']:.4f}"
                if s_["log_loss"] < best_loss:
                    best_loss, self.best_epoch_ = s_["log_loss"], epoch + 1
                    best_state = [(p, k, p[k].copy()) for p, _, k in self._param_list()]
            self.history_["seconds"].append(time.perf_counter() - t0)
            if verbose:
                print(msg + f"  ({self.history_['seconds'][-1]:.1f}s)")
        if best_state is not None:
            for p, k, v in best_state:
                p[k] = v
        return self

In [ ]:
# 梯度檢查：小網路（V = 8、E = 3、H = 4），3 句長度不同（含 padding）
gc = TextRNNNumpy(V=8, E=3, H=4, seed=1)
Xi_gc = np.array([[2, 3, 4, 5, 6], [7, 1, 2, 0, 0], [3, 3, 0, 0, 0]])
len_gc = np.array([5, 3, 2])
y_gc = np.array([0, 1, 1])
gc.loss_and_backward(Xi_gc, len_gc, y_gc)
f_gc = lambda: gc.loss.forward(gc.forward(Xi_gc, len_gc), y_gc)
for layer, name in [(gc.emb, "emb"), (gc.rnn, "lstm"), (gc.fc, "fc")]:
    for k in layer.params:
        numeric = numerical_grad(f_gc, layer.params[k])
        if name == "emb":
            numeric[0] = 0.0
        print(f"✅ {name}.{k:2s} 相對誤差 {rel_error(layer.grads[k], numeric):.1e}（應 < 1e-6）")
print("✅ padding 不改變結果？", np.allclose(gc.forward(Xi_gc[1:2, :3], len_gc[1:2]), gc.forward(Xi_gc, len_gc)[1:2]))

In [ ]:
# 在子集上訓練：同樣的設定，RNN vs LSTM（各 6 epochs；這台電腦 LSTM 一個 epoch 約 25 秒、RNN 約 5 秒）
SUB_TR, SUB_VA, SUB_LEN = 8_000, 1_000, 100
Xs_tr, ls_tr = encode(tok_tr[:SUB_TR], max_len=SUB_LEN)
Xs_va, ls_va = encode(tok_va[:SUB_VA], max_len=SUB_LEN)
ys_tr, ys_va = y_tr[:SUB_TR], y_va[:SUB_VA]
np_models = {}
for cell in ["rnn", "lstm"]:
    print(f"── {cell} ──")
    np_models[cell] = TextRNNNumpy(V, cell=cell).fit(Xs_tr, ls_tr, ys_tr, Xs_va, ls_va, ys_va)
sub_table = pd.DataFrame({cell: val_scores(m.predict_proba(Xs_va, ls_va), ys_va) for cell, m in np_models.items()}).T
sub_table.loc["TF-IDF + LogReg (same 8k/1k subset, last 100 tokens)"] = val_scores(
    make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, token_pattern=r"\S+"), LogisticRegression(C=4, max_iter=3000))
    .fit([" ".join(t[-SUB_LEN:]) for t in tok_tr[:SUB_TR]], ys_tr).predict_proba([" ".join(t[-SUB_LEN:]) for t in tok_va[:SUB_VA]]), ys_va)
display(sub_table)

In [ ]:
# TODO 7（問答）
#
# (1) 梯度檢查：每個參數的相對誤差都 < 1e-6 嗎？
# (2) 子集上的 RNN vs LSTM：validation accuracy 各是多少？train loss 的下降速度呢？跟同樣子集上的 TF-IDF + LogReg 比？
# (3) 子集只有 8,000 篇、100 個字。你預期在完整資料（20,000 篇、200 個字）上，LSTM 和 TF-IDF 的差距會變大還是變小？為什麼？（提示：神經網路需要多少資料才學得會「字向量」）
# 答：

### 3D. 實驗

| 實驗 | 問題 |
| --- | --- |
| 1 | 梯度能傳多遠：$\lVert\partial L/\partial h_t\rVert$、$\lVert\partial L/\partial c_t\rVert$ 隨「離最後一個字幾步」怎麼變？RNN vs LSTM。 |
| 2 | 遺忘閘門的偏差：初始化成 0、1、3，梯度與學習各怎樣？ |
| 3 | 閘門在做什麼：遺忘閘門的值分布；一篇影評裡，哪些字讓閘門開／關？ |
| 4 | 逐字的預測：讀到哪個字時，模型改變心意？ |

In [ ]:
# 實驗 1：梯度能傳多遠（validation 子集裡長度 = 100 的影評，離最後一個字 0～99 步）
full_idx = np.flatnonzero(ls_va == SUB_LEN)[:200]


def grad_profile(model: TextRNNNumpy) -> tuple[np.ndarray, np.ndarray | None]:
    """回傳 (‖∂L/∂h‖ 隨距離, ‖∂L/∂c‖ 隨距離)，都除以「離最後 0 步」的 ‖∂L/∂h‖。"""
    model.loss_and_backward(Xs_va[full_idx], ls_va[full_idx], ys_va[full_idx])
    dh = model.rnn.dh_norms.mean(axis=0)[::-1]  # 反過來：index = 離最後幾步
    dc = model.rnn.dc_norms.mean(axis=0)[::-1] if model.cell == "lstm" else None
    return dh / dh[0], (None if dc is None else dc / dh[0])


fig, ax = plt.subplots(figsize=(9, 4))
for cell, m in np_models.items():
    dh, dc = grad_profile(m)
    ax.semilogy(dh, label=f"{cell} (trained): ‖∂L/∂h‖")
    if dc is not None:
        ax.semilogy(dc, "--", label=f"{cell} (trained): ‖∂L/∂c‖")
    print(f"{cell}：離最後 25、50、99 步時 ‖∂L/∂h‖ 剩", [f"{v:.1e}" for v in dh[[25, 50, 99]]], "" if dc is None else f"；‖∂L/∂c‖ 剩 {[f'{v:.1e}' for v in dc[[25, 50, 99]]]}")
ax.set_xlabel("steps before the last token")
ax.set_ylabel("gradient norm (relative)")
ax.legend()
plt.show()
print("訓練後 RNN 的 Wh 譜半徑：", round(float(np.abs(np.linalg.eigvals(np_models["rnn"].rnn.params["Wh"])).max()), 3),
      "；RNN 的 h 飽和（|h| > 0.95）比例：", round(float((np.abs(np_models["rnn"].rnn.Hs[:, 1:]) > 0.95).mean()), 3))

In [ ]:
# 實驗 2：遺忘閘門偏差 0、1、3（LSTM，同樣的子集與設定）
fb_rows, fb_models = {}, {}
for fb in [0.0, 1.0, 3.0]:
    m = np_models["lstm"] if fb == 1.0 else TextRNNNumpy(V, cell="lstm", forget_bias=fb).fit(Xs_tr, ls_tr, ys_tr, Xs_va, ls_va, ys_va, verbose=False)
    fb_models[fb] = m
    init = TextRNNNumpy(V, cell="lstm", forget_bias=fb)  # 同 seed 的「還沒訓練」版本
    dh0, dc0 = grad_profile(init)
    fb_rows[f"forget_bias={fb}"] = {"init: ‖∂L/∂c‖ at 50 steps": dc0[50], "init: mean f": float(init.rnn.Gates[:, :, init.rnn.H : 2 * init.rnn.H].mean()),
                                    "train loss after epoch 1": m.history_["train_loss"][0], "best val acc": max(m.history_["val_acc"])}
display(pd.DataFrame(fb_rows).T.round(4))

In [ ]:
# 實驗 3：閘門在做什麼（訓練好的 LSTM）
lstm_np = np_models["lstm"]
lstm_np.forward(Xs_va[full_idx], ls_va[full_idx])
G = lstm_np.rnn.Gates
H_np = lstm_np.rnn.H
f_vals, i_vals = G[:, :, H_np : 2 * H_np].ravel(), G[:, :, :H_np].ravel()
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
axes[0].hist(f_vals, bins=50, alpha=0.6, label="forget gate f")
axes[0].hist(i_vals, bins=50, alpha=0.6, label="input gate i")
axes[0].legend()
axes[0].set_title("gate values (200 reviews × 100 steps × H)")
unit_f = G[:, :, H_np : 2 * H_np].mean(axis=(0, 1))
axes[1].bar(np.arange(H_np), np.sort(unit_f)[::-1])
axes[1].set_title("mean forget gate per hidden unit (sorted)")
plt.show()
print(f"f > 0.9 的比例：{np.mean(f_vals > 0.9):.1%}；f < 0.1：{np.mean(f_vals < 0.1):.1%}；平均 f 最大的單元：{unit_f.max():.3f}、最小：{unit_f.min():.3f}")

# 哪些字讓輸入閘門開得最大（平均 i，出現 ≥ 20 次的字）
ID_TO_WORD = {k: w for w, k in WORD_TO_ID.items()}
ID_TO_WORD[1] = "<unk>"
word_ids = Xs_va[full_idx][:, :SUB_LEN].ravel()
mean_i = pd.Series(G[:, :, :H_np].mean(axis=2).ravel()).groupby(word_ids).agg(["mean", "size"])
mean_i = mean_i[mean_i["size"] >= 20].sort_values("mean")
print("輸入閘門平均最小的字：", [ID_TO_WORD[k] for k in mean_i.index[:12]])
print("輸入閘門平均最大的字：", [ID_TO_WORD[k] for k in mean_i.index[-12:]])

In [ ]:
# 實驗 4：逐字的預測（用每一步的 h_t 預測正面的機率）
def prefix_pos_prob(model: TextRNNNumpy, tokens: list[str]) -> np.ndarray:
    Xi, L = encode([tokens], max_len=len(tokens))
    model.forward(Xi, L)
    Z = model.rnn.Hs[0, 1:] @ model.fc.params["W"] + model.fc.params["b"]
    P = np.exp(Z - Z.max(axis=1, keepdims=True))
    return (P / P.sum(axis=1, keepdims=True))[:, 1]


demo = [
    "this movie was great . the acting was wonderful and the story was moving .",
    "i expected a great movie , but it was boring and the acting was terrible .",
    "the first half was terrible , but the ending was brilliant and i loved it .",
    "not bad at all . i was surprised .",
]
fig, axes = plt.subplots(len(demo), 1, figsize=(15, 2.4 * len(demo)))
for ax, sent in zip(axes, demo):
    toks = sent.split()
    p = prefix_pos_prob(lstm_np, toks)
    ax.plot(p, "o-")
    ax.axhline(0.5, color="gray", ls=":")
    ax.set_xticks(np.arange(len(toks)), toks, rotation=30, fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_ylabel("P(pos)")
fig.tight_layout()
plt.show()

In [ ]:
# TODO 8（問答）實驗 1–4
#
# (1) 實驗 1：
#     (a) 離最後 50、99 步時，RNN 的 ‖∂L/∂h‖ 剩多少？LSTM 的 ‖∂L/∂h‖、‖∂L/∂c‖ 呢？
#     (b) LSTM 的 ‖∂L/∂c‖ 衰減得比 ‖∂L/∂h‖ 慢嗎？用 3B 表格的最後一行解釋。
#     (c) 這對「影評開頭的字能不能影響參數更新」有什麼意義？
#     (d) 如果訓練後 RNN 的梯度「沒有消失」（甚至變大）：看它的 Wh 譜半徑（前置複習：第 2 課實驗 1 的 ρ = 1.5、3 曲線）。
#         梯度沒有消失，RNN 為什麼還是比 LSTM 差？（提示：梯度很大 ≠ 梯度有用 —— 爆炸的梯度方向很不穩定、要靠裁剪壓住；
#         而且這是「平均」的範數，看不出是不是只有少數幾個單元在放大）
# 答：
#
# (2) 實驗 2：
#     (a) 初始化時的平均 f 是多少（σ(0)、σ(1)、σ(3)）？50 步之後的 ‖∂L/∂c‖ 差幾個數量級？
#     (b) 訓練的結果（第 1 個 epoch 的 loss、最好的 val acc）有差別嗎？如果差別不大，為什麼？（提示：訓練過程中 f 自己會學）
# 答：
#
# (3) 實驗 3：
#     (a) 遺忘閘門的值大多接近 0、1，還是中間？有沒有「幾乎永遠記住」（平均 f 很大）的單元？
#         跟初始化時的 σ(1) ≈ 0.73 比，訓練後變了多少？在 100 個字、8,000 篇的子集上，模型可能主要靠 c 那條「長期記憶」，還是靠最後幾個字？
#     (b) 輸入閘門開最大／最小的字各是哪些？跟情緒有關嗎？（提示：標點、停用詞 vs 形容詞）
# 答：
#
# (4) 實驗 4：「but」之後模型有沒有改變心意？「not bad」呢？哪一個句子模型判斷錯了？為什麼詞袋模型很難處理這種句子？
# 答：

---
## 4. 套件實作：PyTorch `nn.LSTM`

| PyTorch | 對應手寫的 | 注意 |
| --- | --- | --- |
| `nn.LSTM(E, H, batch_first=True)` | `LSTM` | `weight_ih_l0` = $W_x^\top$ $(4H, E)$、`weight_hh_l0` = $W_h^\top$；四段的順序也是 **$i, f, g, o$**；偏差兩個（`bias_ih_l0` + `bias_hh_l0`） |
| 回傳 `output, (h_n, c_n)` | `Hs`、`Cs` | 多了一個 $c_n$（RNN 只有 $h_n$） |
| `num_layers=2` | — | Stacked LSTM：第 1 層每一步的 $h_t$ 當成第 2 層的輸入 $x_t$ |
| `bidirectional=True` | — | 另一個 LSTM 從右往左讀；$h_n$ 兩個方向串起來 → $2H$ |
| `pack_padded_sequence` | mask | 同第 2 課 |

PyTorch 預設的遺忘閘門偏差是 **0**（`bias_ih`、`bias_hh` 都是均勻分布的隨機數）。下面的 `TextRNNTorch` 把它設成 1（兩個偏差的 $f$ 段加起來 = 1），跟手寫版一致。

**本機用 CPU**：同第 2 課。這一章的模型較大（約 130 萬個參數，大部分在 Embedding），單向 LSTM 一個 epoch 在這台電腦約 30 秒（雙向、兩層約加倍）；Mac 的 MPS 可能比較快，可以把 `device` 改成 `get_device()` 試試。

In [ ]:
import torch
from torch import nn
from ml_dl_llm.utils import count_parameters, get_device, set_seed

device = torch.device("cpu")


class TextRNNTorch(nn.Module):
    """Embedding → nn.RNN／nn.LSTM（可多層、可雙向）→ Linear(2)。forward(X, lengths) → logits (n, 2)。"""

    def __init__(self, V: int, cell: str = "lstm", E: int = 64, H: int = 64, num_layers: int = 1, bidirectional: bool = False,
                 forget_bias: float = 1.0) -> None:
        super().__init__()
        self.emb = nn.Embedding(V, E, padding_idx=0)
        RNNClass = {"rnn": nn.RNN, "lstm": nn.LSTM}[cell]
        self.rnn = RNNClass(E, H, num_layers=num_layers, batch_first=True, bidirectional=bidirectional)
        self.fc = nn.Linear(H * (2 if bidirectional else 1), 2)
        if cell == "lstm":
            with torch.no_grad():
                for name, p in self.rnn.named_parameters():
                    if name.startswith("bias_ih"):
                        p[H : 2 * H] = forget_bias
                    elif name.startswith("bias_hh"):
                        p[H : 2 * H] = 0.0

    def forward(self, X: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        packed = nn.utils.rnn.pack_padded_sequence(self.emb(X), lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.rnn(packed)
        if isinstance(h_n, tuple):  # LSTM 回傳 (h_n, c_n)
            h_n = h_n[0]
        h = torch.cat([h_n[-2], h_n[-1]], dim=1) if self.rnn.bidirectional else h_n[-1]  # 最上面一層
        return self.fc(h)


def load_from_numpy(net: TextRNNTorch, model: TextRNNNumpy) -> TextRNNTorch:
    """把手寫的 LSTM 權重複製進 PyTorch（轉置；bias_hh 設 0）。"""
    with torch.no_grad():
        net.emb.weight.copy_(torch.as_tensor(model.emb.params["W"]))
        net.rnn.weight_ih_l0.copy_(torch.as_tensor(model.rnn.params["Wx"].T))
        net.rnn.weight_hh_l0.copy_(torch.as_tensor(model.rnn.params["Wh"].T))
        net.rnn.bias_ih_l0.copy_(torch.as_tensor(model.rnn.params["b"]))
        net.rnn.bias_hh_l0.zero_()
        net.fc.weight.copy_(torch.as_tensor(model.fc.params["W"].T))
        net.fc.bias.copy_(torch.as_tensor(model.fc.params["b"]))
    return net


# 核對：同樣的權重（float64）、同一個 batch（含 padding）
idx_chk = np.arange(16)
loss_np = lstm_np.loss_and_backward(Xs_va[idx_chk], ls_va[idx_chk], ys_va[idx_chk])
net_chk = load_from_numpy(TextRNNTorch(V, E=32, H=64).double(), lstm_np)
T_chk = int(ls_va[idx_chk].max())
loss_t = nn.functional.cross_entropy(net_chk(torch.as_tensor(Xs_va[idx_chk, :T_chk]), torch.as_tensor(ls_va[idx_chk])), torch.as_tensor(ys_va[idx_chk]))
loss_t.backward()
print(f"✅ loss：手寫 {loss_np:.10f}、PyTorch {loss_t.item():.10f}")
for k, (a, b) in {
    "emb.W": (lstm_np.emb.grads["W"], net_chk.emb.weight.grad.numpy()),
    "lstm.Wx": (lstm_np.rnn.grads["Wx"], net_chk.rnn.weight_ih_l0.grad.numpy().T),
    "lstm.Wh": (lstm_np.rnn.grads["Wh"], net_chk.rnn.weight_hh_l0.grad.numpy().T),
    "lstm.b": (lstm_np.rnn.grads["b"], net_chk.rnn.bias_ih_l0.grad.numpy()),
    "fc.W": (lstm_np.fc.grads["W"], net_chk.fc.weight.grad.numpy().T),
}.items():
    print(f"✅ {k:8s} 梯度最大差異 {np.abs(a - b).max():.1e}（應 < 1e-10）")

#### 用 PyTorch 訓練

`train_seq` 跟第 2 課相同（Adam、梯度裁剪 5、每個 epoch 量 validation、`restore_best`），只加了一件事：batch **依長度分桶**（先依長度排序切成 batch，再打亂 batch 的順序）—— 影評長短差很多，隨機湊一個 batch 會補很多 padding。

先比較同樣大小（$E = H = 64$）的 RNN 與 LSTM，各 5 個 epoch。

In [ ]:
@torch.no_grad()
def predict_proba_seq(net: nn.Module, Xi: np.ndarray, lengths: np.ndarray, batch_size: int = 1000) -> np.ndarray:
    """(n, 2) 的類別機率（float64 numpy）。"""
    net.eval()
    out = []
    for s in range(0, len(Xi), batch_size):
        L = lengths[s : s + batch_size]
        X = torch.as_tensor(Xi[s : s + batch_size, : L.max()], device=device)
        out.append(torch.softmax(net(X, torch.as_tensor(L)).double(), dim=1).cpu().numpy())
    return np.vstack(out)


def train_seq(net: nn.Module, Xi: np.ndarray, lengths: np.ndarray, y: np.ndarray, Xi_val: np.ndarray | None = None,
              len_val: np.ndarray | None = None, y_val: np.ndarray | None = None, *, lr: float = 2e-3, batch_size: int = 64,
              n_epochs: int = 5, clip: float = 5.0, seed: int = 0, restore_best: bool = True, verbose: bool = True) -> dict:
    """Adam + 梯度裁剪 + 依長度分桶。history 同第 2 課。"""
    set_seed(seed)
    net.to(device)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    order = np.argsort(lengths, kind="stable")
    batches = [order[s : s + batch_size] for s in range(0, len(order), batch_size)]
    rng = np.random.default_rng(seed)
    hist: dict = {"train_loss": [], "val_loss": [], "val_acc": [], "seconds": []}
    best_loss, best_epoch, best_state = np.inf, 0, None
    for epoch in range(n_epochs):
        net.train()
        t0, total = time.perf_counter(), 0.0
        for b in rng.permutation(len(batches)):
            idx = batches[b]
            L = lengths[idx]
            loss = loss_fn(net(torch.as_tensor(Xi[idx, : L.max()], device=device), torch.as_tensor(L)), torch.as_tensor(y[idx], device=device))
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), clip)
            opt.step()
            total += loss.item() * len(idx)
        hist["train_loss"].append(total / len(y))
        msg = f"epoch {epoch + 1:2d}  train_loss {total / len(y):.4f}"
        if Xi_val is not None:
            s_ = val_scores(predict_proba_seq(net, Xi_val, len_val), y_val)
            hist["val_loss"].append(s_["log_loss"])
            hist["val_acc"].append(s_["accuracy"])
            msg += f"  val_loss {s_['log_loss']:.4f}  val_acc {s_['accuracy']:.4f}"
            if s_["log_loss"] < best_loss:
                best_loss, best_epoch = s_["log_loss"], epoch + 1
                best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
        hist["seconds"].append(time.perf_counter() - t0)
        if verbose:
            print(msg + f"  ({hist['seconds'][-1]:.0f}s)")
    hist["best_epoch"], hist["best_val_loss"] = best_epoch, best_loss
    if restore_best and best_state is not None:
        net.load_state_dict(best_state)
    return hist


torch_models, torch_hists = {}, {}
for cell in ["rnn", "lstm"]:
    print(f"── {cell} ──")
    set_seed(0)
    net = TextRNNTorch(V, cell=cell)
    torch_hists[cell] = train_seq(net, Xi_tr, len_tr, y_tr, Xi_va, len_va, y_va)
    torch_models[cell] = net
    val_table.loc[f"{cell.upper()} (PyTorch, E=H=64, last {MAX_LEN} tokens)"] = val_scores(predict_proba_seq(net, Xi_va, len_va))
print("LSTM 參數量：", count_parameters(torch_models["lstm"]), "；其中 Embedding：", torch_models["lstm"].emb.weight.numel())
display(val_table)

In [ ]:
# TODO 9（問答）PyTorch
#
# (1) 核對：loss 與梯度的差異是多少？PyTorch 的四段順序（i, f, g, o）跟手寫的一樣嗎？如果順序弄錯（例如 f、i 對調），核對會怎樣？
# 答：
#
# (2) RNN vs LSTM（同樣的 E、H、epoch）：
#     (a) validation accuracy 各是多少？RNN 的 train loss 有沒有卡住？跟 TODO 2 (3)(a) 的預測比。
#     (b) 參數量：LSTM 的循環層是 RNN 的幾倍（提示：4 段）？整個模型呢？大部分參數在哪裡？
#     (c) 每個 epoch 的時間差多少？
# 答：

---
## 5. 調參

| 類型 | 超參數 | 這一節試的值 |
| --- | --- | --- |
| 資料 | 截斷長度 `max_len`、留開頭還是結尾 | 100、200、400（tail）；200 head |
| 容量 | 雙向、層數 | 單向 1 層、雙向 1 層、單向 2 層 |
| 最佳化 | lr、epoch | Adam 2e-3；5 epochs + `restore_best` |

每個 LSTM 都是 $E = H = 64$。（訓練慢，每個設定只跑一個 seed；5C 再用 3 個 seed 估計雜訊。）

In [ ]:
# 5A. 截斷：max_len × side（單向 LSTM）
trunc_rows = []
enc_cache = {}
for max_len, side in [(100, "tail"), (200, "tail"), (400, "tail"), (200, "head")]:
    a, la = encode(tok_tr, max_len, side)
    b, lb = encode(tok_va, max_len, side)
    enc_cache[(max_len, side)] = (a, la, b, lb)
    if (max_len, side) == (MAX_LEN, "tail"):
        net, h = torch_models["lstm"], torch_hists["lstm"]
    else:
        set_seed(0)
        net = TextRNNTorch(V)
        h = train_seq(net, a, la, y_tr, b, lb, y_va, verbose=False)
    trunc_rows.append({"max_len": max_len, "side": side, **val_scores(predict_proba_seq(net, b, lb)), "best_epoch": h["best_epoch"], "sec/epoch": np.mean(h["seconds"])})
trunc_tbl = pd.DataFrame(trunc_rows)
display(trunc_tbl)

In [ ]:
# 5B. 結構：雙向、兩層（max_len = 200、tail）
arch_rows = [{"arch": "LSTM 1 layer", **val_scores(predict_proba_seq(torch_models["lstm"], Xi_va, len_va)), "params": count_parameters(torch_models["lstm"]),
              "best_epoch": torch_hists["lstm"]["best_epoch"], "sec/epoch": np.mean(torch_hists["lstm"]["seconds"])}]
for name, kw in {"Bi-LSTM 1 layer": dict(bidirectional=True), "LSTM 2 layers": dict(num_layers=2)}.items():
    set_seed(0)
    net = TextRNNTorch(V, **kw)
    h = train_seq(net, Xi_tr, len_tr, y_tr, Xi_va, len_va, y_va, verbose=False)
    torch_models[name], torch_hists[name] = net, h
    arch_rows.append({"arch": name, **val_scores(predict_proba_seq(net, Xi_va, len_va)), "params": count_parameters(net), "best_epoch": h["best_epoch"], "sec/epoch": np.mean(h["seconds"])})
arch_tbl = pd.DataFrame(arch_rows).set_index("arch")
display(arch_tbl)

In [ ]:
# 5C. 選設定（one-standard-error rule）＋ seed 雜訊
# validation 5,000 篇，accuracy ≈ 0.86 時的標準誤 = sqrt(0.86 × 0.14 / 5000) ≈ 0.005。
# 跟最好的差距在一個標準誤以內的設定都「分不出好壞」→ 選其中最快的（每個 epoch 秒數最少）。
cands = [((r["max_len"], r["side"], "LSTM 1 layer"), r["accuracy"], r["best_epoch"], r["sec/epoch"]) for r in trunc_rows] + \
        [((MAX_LEN, "tail", a), r["accuracy"], r["best_epoch"], r["sec/epoch"]) for a, r in arch_tbl.iterrows()]
cands = list({c[0]: c for c in cands}.values())  # 「200 tail 1 層」在 5A、5B 各出現一次，只留一筆
SE = float(np.sqrt(0.86 * 0.14 / len(y_va)))
top_acc = max(c[1] for c in cands)
within = [c for c in cands if c[1] >= top_acc - SE]
print(f"最好的 val accuracy {top_acc:.4f}；一個標準誤 {SE:.4f} 以內的設定：", [(c[0], c[1], round(c[3], 1)) for c in within])
(best_len, best_side, best_arch), acc_seed0, epoch_seed0, _sec = min(within, key=lambda c: c[3])
ARCH_KW = {"LSTM 1 layer": {}, "Bi-LSTM 1 layer": dict(bidirectional=True), "LSTM 2 layers": dict(num_layers=2)}
best_cfg = {"max_len": best_len, "side": best_side, "arch": best_arch}
print("選中的設定（一個標準誤內最快的）：", best_cfg)
a, la, b, lb = enc_cache.get((best_len, best_side)) or (*encode(tok_tr, best_len, best_side), *encode(tok_va, best_len, best_side))
seed_rows = []
for seed in [1, 2]:
    set_seed(seed)
    net = TextRNNTorch(V, **ARCH_KW[best_arch])
    h = train_seq(net, a, la, y_tr, b, lb, y_va, seed=seed, verbose=False)
    seed_rows.append({"seed": seed, **val_scores(predict_proba_seq(net, b, lb)), "best_epoch": h["best_epoch"]})
seed_tbl = pd.DataFrame(seed_rows).set_index("seed")
display(seed_tbl)
print("3 個 seed（含原本 seed 0）的 accuracy：", [float(acc_seed0)] + [float(v) for v in seed_tbl["accuracy"]])
best_cfg["epochs"] = int(round(np.mean(list(seed_tbl["best_epoch"]) + [epoch_seed0])))

In [ ]:
# TODO 10（問答）調參
#
# (1) 5A：
#     (a) max_len 100 → 200 → 400，accuracy 怎麼變？時間呢？值得嗎？
#     (b) 同樣 200 個字，head 和 tail 哪個好？跟 EDA（TODO 1 (4)）的結論一樣嗎？
#     (c) 第 2 節：TF-IDF 只看最後 200 個字時掉了多少？LSTM 從 200 → 400 進步多少？這兩個數字說明了什麼？
# 答：
#
# (2) 5B：雙向、兩層各讓 accuracy 變多少？參數量與時間呢？用 5C 的 seed 雜訊判斷：差距是真的嗎？
# 答：
#
# (3) 5C：
#     (a) 一個標準誤以內有哪些設定？選中的是哪一個？如果只看「最高分」會選哪一個？兩者差多少、時間差幾倍？
#     (b) 3 個 seed 的 accuracy 差多少？跟一個標準誤（0.005）比呢？最好的 epoch 一樣嗎？
# 答：

---
## 6. 測試集評估

依 README：用最佳設定在**完整訓練集**（train + validation = 25,000 篇）refit（字表維持用 train 的 20,000 篇建的那一份），epoch 數用 5C 的 best epoch 平均，測試集只評估一次。

參照點：TF-IDF + LogReg 也在 25,000 篇上 refit；RNN、LSTM（PyTorch，第 4 節）與手寫 NumPy LSTM（8,000 篇子集訓練）用的是原本的版本。

In [ ]:
from ml_dl_llm.metrics import classification_metrics
from ml_dl_llm.results import compare, log_result

BENCH = "imdb"
print("最終設定：", best_cfg)
Xa_full, la_full = encode(tok_tr + tok_va, best_cfg["max_len"], best_cfg["side"])
y_trva = np.concatenate([y_tr, y_va])
set_seed(0)
net_final = TextRNNTorch(V, **ARCH_KW[best_cfg["arch"]])
train_seq(net_final, Xa_full, la_full, y_trva, n_epochs=best_cfg["epochs"], verbose=False)
Xt_best, lt_best = encode(test_tok, best_cfg["max_len"], best_cfg["side"])
tfidf_full = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200_000, token_pattern=r"\S+"), LogisticRegression(C=4, max_iter=3000)).fit(
    docs_tr + docs_va, y_trva)
Xs_te, ls_te = encode(test_tok, max_len=SUB_LEN)

eval_models = {
    "TF-IDF 1-2-gram + LogReg (train+val)": (tfidf_full.predict_proba([" ".join(t) for t in test_tok]), {"ngram": [1, 2], "C": 4}),
    "LSTM (scratch NumPy, 8k subset, last 100 tokens)": (lstm_np.predict_proba(Xs_te, ls_te), {"E": 32, "H": 64, "train": 8000, "max_len": 100, "optimizer": "adam"}),
    "RNN (PyTorch, last 200 tokens)": (predict_proba_seq(torch_models["rnn"], Xi_te, len_te), {"E": 64, "H": 64, "max_len": MAX_LEN}),
    "LSTM (PyTorch, last 200 tokens)": (predict_proba_seq(torch_models["lstm"], Xi_te, len_te), {"E": 64, "H": 64, "max_len": MAX_LEN}),
    "LSTM tuned (PyTorch, train+val)": (predict_proba_seq(net_final, Xt_best, lt_best), {"E": 64, "H": 64, **best_cfg}),
}
test_scores = {}
for name, (prob, params) in eval_models.items():
    m = classification_metrics(y_test, prob.argmax(axis=1), prob[:, 1])
    m["log_loss"] = round(log_loss(y_test, np.clip(prob, 1e-12, 1), labels=[0, 1]), 4)
    test_scores[name] = m
    log_result(BENCH, name, m, **params)
display(pd.DataFrame(test_scores).T)
compare(BENCH, "accuracy")

---
## 7. 診斷

1. **準確率 vs 影評長度**：LSTM 只看前 `max_len` 個字；長的影評被截斷，會不會比較差？TF-IDF 看整篇。
2. **兩個模型意見不同的影評**：LSTM 對、TF-IDF 錯的長什麼樣子？反過來呢？
3. **最有把握地錯**的影評。

In [ ]:
prob_lstm = eval_models["LSTM tuned (PyTorch, train+val)"][0]
prob_tfidf = eval_models["TF-IDF 1-2-gram + LogReg (train+val)"][0]
pred_lstm, pred_tfidf = prob_lstm.argmax(axis=1), prob_tfidf.argmax(axis=1)
test_lens = np.array([len(t) for t in test_tok])
bins = [0, 100, 200, 300, 500, 800, 5000]
len_bin = pd.cut(test_lens, bins)
by_len = pd.DataFrame({"LSTM tuned": pred_lstm == y_test, "TF-IDF": pred_tfidf == y_test, "bin": len_bin}).groupby("bin", observed=True).agg(["mean", "size"])
display(by_len.round(3))

both = pd.crosstab(pd.Series(pred_lstm == y_test, name="LSTM correct"), pd.Series(pred_tfidf == y_test, name="TF-IDF correct"))
display(both)
ens = (prob_lstm + prob_tfidf) / 2
print(f"兩者平均（ensemble）的 test accuracy：{accuracy_score(y_test, ens.argmax(axis=1)):.4f}")


def show(idx: np.ndarray, title: str, k: int = 3) -> None:
    print(f"\n==== {title} ====")
    for i in idx[:k]:
        print(f"[label {y_test[i]}；P(pos) LSTM {prob_lstm[i, 1]:.2f}、TF-IDF {prob_tfidf[i, 1]:.2f}；{test_lens[i]} 字]")
        print("   ", " ".join(test_tok[i][:80]), "..." if test_lens[i] > 80 else "")


rng_show = np.random.default_rng(0)
show(rng_show.permutation(np.flatnonzero((pred_lstm == y_test) & (pred_tfidf != y_test))), "LSTM 對、TF-IDF 錯")
show(rng_show.permutation(np.flatnonzero((pred_lstm != y_test) & (pred_tfidf == y_test))), "LSTM 錯、TF-IDF 對")
wrong = np.flatnonzero(pred_lstm != y_test)
show(wrong[np.argsort(-prob_lstm[wrong].max(axis=1))], "LSTM 最有把握地錯")

In [ ]:
# TODO 11（問答，本章總結）
#
# (1) 測試集：
#     (a) 依 accuracy 排序 TF-IDF、RNN、LSTM、LSTM tuned、手寫 LSTM。跟你在 TODO 2 (3) 的預測一樣嗎？
#     (b) RNN → LSTM 進步多少？這個進步主要是因為哪件事？（實驗 1、TODO 5）
#     (c) 如果 LSTM 沒有贏過 TF-IDF：影評情感分類的哪些性質讓「詞袋」就很強？（提示：情緒字、整篇都看得到、資料量）
#     (d) LSTM tuned（train + val refit）跟只在 train 上訓練、有 early stopping 的 LSTM 比，test 差多少？
#         refit 用了更多資料，為什麼不保證比較好？（提示：refit 時沒有 validation 可以 early stopping，epoch 數是估的；seed 雜訊）
# 答：
#
# (2) 長度：
#     (a) 長的影評（> max_len）LSTM 的準確率有下降嗎？TF-IDF 呢？
#     (b) 兩個模型「一個對一個錯」的影評有多少篇？平均兩者的機率（ensemble）之後進步多少？為什麼兩個不同類型的模型平均起來會更好？
# 答：
#
# (3) 看例子：LSTM 對、TF-IDF 錯的影評有什麼共同點（轉折、否定、反諷）？LSTM 最有把握地錯的呢？
# 答：
#
# (4) 下一課 GRU：把 LSTM 的三個閘門精簡成兩個、把 c 和 h 合併。你預期它在這份資料上會比 LSTM 好、差、還是差不多？參數量呢？
# 答：

---
## 8. 本章小結

（完成所有 TODO、並經過回饋與修正之後，依 README「章末小結」填寫：問答註解轉成 markdown、結果表、核心觀念、待補強。）

### 方法與結果

IMDB（train 20,000／validation 5,000／test 25,000）

| 模型 | 做法 | 關鍵性質 | val acc | test acc | test AUC |
| --- | --- | --- | --- | --- | --- |
| Dummy prior | | | | — | — |
| BoW + LogReg | | | | — | — |
| TF-IDF 1～2-gram + LogReg | | | | | |
| LSTM（手寫 NumPy，子集） | | | | | |
| RNN（PyTorch） | | | | | |
| LSTM（PyTorch） | | | | | |
| LSTM tuned | | | | | |

### 核心觀念

1.
2.
3.

### 待補強（之後回來可以學）

**理論**
- ☐ LSTM 梯度的完整分析（為什麼 cell state 那條路不會消失；peephole 連接）
- ☐ 遺忘閘門偏差與「chrono initialization」（Tallec & Ollivier 2018）
- ☐ 截斷 BPTT（truncated BPTT）用於很長的序列

**延伸**
- ☐ 預訓練詞向量（word2vec、GloVe）取代從零學的 Embedding
- ☐ Dropout 用在 RNN（variational dropout）、AWD-LSTM
- ☐ 下一課 GRU；Attention（第四部分 Seq2Seq＋Attention）讓模型直接「回頭看」任何一個字
- ☐ 用 Transformer／BERT（第五部分）做同一個任務，比較 LSTM